# 01.7 - Graphs

## What is a graph?
A set of **vertices (nodes)** connected by **edges**. Trees and linked lists are special graphs.

```
   A --- B          Undirected: friendship (A-B both ways)
   |   / |
   |  /  |          Directed:  A -> B  (Twitter "follows", web links)
   C --- D          Weighted:  edge has a cost (km, minutes, price)
```
| Term | Meaning |
|---|---|
| **Directed / undirected** | One-way vs two-way edges |
| **Weighted** | Edges carry a cost |
| **Degree** | Number of edges at a node (in-degree / out-degree if directed) |
| **Path / cycle** | Sequence of edges / path returning to its start |
| **Connected component** | Group of nodes all reachable from each other |
| **DAG** | Directed Acyclic Graph - dependencies, tasks, Git history |

## Where are graphs used?
| Real system | Nodes / edges |
|---|---|
| **Google Maps, Uber, Bolt, airlines** | Intersections / roads (weighted) → shortest route |
| **Facebook, LinkedIn, Instagram** | People / friendships → suggestions, degrees of separation |
| **The Internet & routers** | Routers / cables → packet routing (OSPF = Dijkstra) |
| **Google PageRank, web crawlers** | Pages / links |
| **Recommendations (Netflix, Amazon)** | Users & items / interactions |
| **Package managers (pip, npm), build tools, Excel formulas** | Packages / dependencies → **topological sort** |
| **Git** | Commits / parent links (a DAG) |
| **Fraud detection (banks, mobile money)** | Accounts / transfers → find suspicious rings |
| **Knowledge graphs, chatbots, biology networks** | Concepts / relations |
| **Games & robotics** | Grid cells / moves → pathfinding (A*) |

> **A grid is a graph too:** each cell is a node, connected to its 4 neighbours.

## 1. Three ways to represent a graph
| Representation | Space | Check edge u–v | List neighbours | Use when |
|---|---|---|---|---|
| **Adjacency list** `{u: [v, ...]}` | O(V + E) | O(deg) | O(deg) | **Almost always** (sparse graphs) |
| Adjacency matrix `M[u][v]` | O(V²) | O(1) | O(V) | Dense graphs, small V |
| Edge list `[(u, v, w), ...]` | O(E) | O(E) | O(E) | Kruskal's MST, input format |

In [1]:
from collections import defaultdict, deque

edges = [("A", "B"), ("A", "C"), ("B", "C"), ("B", "D"), ("C", "D"), ("D", "E")]

# Adjacency list (undirected)
adj = defaultdict(list)
for u, v in edges:
    adj[u].append(v)
    adj[v].append(u)
print("adjacency list:", dict(adj))

# Adjacency matrix
nodes = sorted(adj)
idx = {n: i for i, n in enumerate(nodes)}
matrix = [[0] * len(nodes) for _ in nodes]
for u, v in edges:
    matrix[idx[u]][idx[v]] = matrix[idx[v]][idx[u]] = 1
print("   ", nodes)
for n, row in zip(nodes, matrix):
    print(n, " ", row)

adjacency list: {'A': ['B', 'C'], 'B': ['A', 'C', 'D'], 'C': ['A', 'B', 'D'], 'D': ['B', 'C', 'E'], 'E': ['D']}
    ['A', 'B', 'C', 'D', 'E']
A   [0, 1, 1, 0, 0]
B   [1, 0, 1, 1, 0]
C   [1, 1, 0, 1, 0]
D   [0, 1, 1, 0, 1]
E   [0, 0, 0, 1, 0]


## 2. A reusable Graph class with BFS & DFS
- **BFS** (queue): explores level by level → **shortest path in unweighted graphs**.
- **DFS** (stack / recursion): goes deep first → connectivity, cycles, paths, topological sort.
Both are **O(V + E)**. Always keep a **visited** set, or you'll loop forever on cycles.

In [2]:
class Graph:
    def __init__(self, directed=False):
        self.adj = defaultdict(list)
        self.directed = directed

    def add_edge(self, u, v):
        self.adj[u].append(v)
        if not self.directed:
            self.adj[v].append(u)

    def bfs(self, start):
        visited, order, q = {start}, [], deque([start])
        while q:
            node = q.popleft()
            order.append(node)
            for nei in self.adj[node]:
                if nei not in visited:
                    visited.add(nei)
                    q.append(nei)
        return order

    def dfs(self, start):
        visited, order = set(), []
        def visit(node):
            visited.add(node)
            order.append(node)
            for nei in self.adj[node]:
                if nei not in visited:
                    visit(nei)
        visit(start)
        return order

    def shortest_path(self, start, goal):       # unweighted -> BFS
        parent, q = {start: None}, deque([start])
        while q:
            node = q.popleft()
            if node == goal:
                path = []
                while node is not None:
                    path.append(node)
                    node = parent[node]
                return path[::-1]
            for nei in self.adj[node]:
                if nei not in parent:
                    parent[nei] = node
                    q.append(nei)
        return None

g = Graph()
for u, v in edges:
    g.add_edge(u, v)
print("BFS from A:", g.bfs("A"))
print("DFS from A:", g.dfs("A"))
print("Shortest A -> E:", g.shortest_path("A", "E"))
assert len(g.shortest_path("A", "E")) == 4

BFS from A: ['A', 'B', 'C', 'D', 'E']
DFS from A: ['A', 'B', 'C', 'D', 'E']
Shortest A -> E: ['A', 'B', 'D', 'E']


In [3]:
# Real-world demo: degrees of separation in a social network (LinkedIn "2nd connection")
friends = Graph()
for a, b in [("You", "Amina"), ("You", "Jean"), ("Amina", "Eric"), ("Jean", "Grace"),
             ("Eric", "Sam"), ("Grace", "Sam"), ("Sam", "Recruiter@Google")]:
    friends.add_edge(a, b)
path = friends.shortest_path("You", "Recruiter@Google")
print(" -> ".join(path), f"({len(path) - 2} people between you)")

You -> Amina -> Eric -> Sam -> Recruiter@Google (3 people between you)


---
## Solved problems

### 1. Find if Path Exists in Graph - LC #1971 (Easy)
Undirected graph with `n` nodes and `edges`. Is there a path from `source` to `destination`?

**Idea:** Build adjacency list, then iterative DFS with a stack.

In [4]:
def valid_path(n, edges, source, destination):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    stack, seen = [source], {source}
    while stack:
        node = stack.pop()
        if node == destination:
            return True
        for nei in adj[node]:
            if nei not in seen:
                seen.add(nei)
                stack.append(nei)
    return False

assert valid_path(3, [[0, 1], [1, 2], [2, 0]], 0, 2)
assert not valid_path(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5)
print("Path Exists ")

Path Exists 


### 2. Max Area of Island - LC #695 (Medium)
Largest connected group of 1s in a grid (4-directional).

**Idea:** DFS flood-fill from every unvisited land cell; the DFS returns the size of its island.

In [5]:
def max_area_of_island(grid):
    rows, cols = len(grid), len(grid[0])
    def dfs(r, c):
        if r < 0 or c < 0 or r >= rows or c >= cols or grid[r][c] != 1:
            return 0
        grid[r][c] = 0                       # mark visited by sinking the land
        return 1 + dfs(r + 1, c) + dfs(r - 1, c) + dfs(r, c + 1) + dfs(r, c - 1)
    return max((dfs(r, c) for r in range(rows) for c in range(cols)), default=0)

grid = [[0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
        [0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0],
        [0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]]
assert max_area_of_island(grid) == 6
print("Max Area of Island ")

Max Area of Island 


### 3. Clone Graph - LC #133 (Medium) Deep-copy a connected undirected graph.

**Idea:** DFS with a hash map `original → copy`; the map doubles as the visited set and handles cycles.

In [6]:
class Node:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []

def clone_graph(node):
    if not node:
        return None
    copies = {}                      # original -> copy (also our visited set)
    def dfs(n):
        if n in copies:
            return copies[n]
        copy = Node(n.val)
        copies[n] = copy
        for nei in n.neighbors:
            copy.neighbors.append(dfs(nei))
        return copy
    return dfs(node)

# square 1-2-3-4-1
n1, n2, n3, n4 = Node(1), Node(2), Node(3), Node(4)
n1.neighbors = [n2, n4]; n2.neighbors = [n1, n3]; n3.neighbors = [n2, n4]; n4.neighbors = [n1, n3]
c = clone_graph(n1)
assert c is not n1 and c.val == 1 and [x.val for x in c.neighbors] == [2, 4]
assert c.neighbors[0].neighbors[1].val == 3 and c.neighbors[0].neighbors[1] is not n3
print("Clone Graph ")

Clone Graph 


### 4. Pacific Atlantic Water Flow - LC #417 (Medium)
Cells from which water can flow to both the Pacific (top/left edges) and Atlantic (bottom/right edges).

**Idea:** Reverse thinking: BFS **from the oceans inward** (moving to equal or higher cells). Answer = cells reached by both searches.

In [7]:
def pacific_atlantic(heights):
    rows, cols = len(heights), len(heights[0])

    def bfs(starts):
        seen, q = set(starts), deque(starts)
        while q:
            r, c = q.popleft()
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if (0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in seen
                        and heights[nr][nc] >= heights[r][c]):     # flowing "uphill" backwards
                    seen.add((nr, nc))
                    q.append((nr, nc))
        return seen

    pacific = bfs([(0, c) for c in range(cols)] + [(r, 0) for r in range(rows)])
    atlantic = bfs([(rows - 1, c) for c in range(cols)] + [(r, cols - 1) for r in range(rows)])
    return sorted([list(cell) for cell in pacific & atlantic])

h = [[1, 2, 2, 3, 5], [3, 2, 3, 4, 4], [2, 4, 5, 3, 1], [6, 7, 1, 4, 5], [5, 1, 1, 2, 4]]
assert pacific_atlantic(h) == [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
print("Pacific Atlantic ")

Pacific Atlantic 


### 5. Is Graph Bipartite? - LC #785 (Medium)
Can nodes be split into two groups with every edge going between groups? (Scheduling, matching, 2-colouring maps.)

**Idea:** BFS colouring with 2 colours; a neighbour with the same colour means it's impossible. Loop over all nodes for disconnected graphs.

In [8]:
def is_bipartite(graph):
    color = {}
    for start in range(len(graph)):
        if start in color:
            continue
        color[start] = 0
        q = deque([start])
        while q:
            node = q.popleft()
            for nei in graph[node]:
                if nei not in color:
                    color[nei] = 1 - color[node]
                    q.append(nei)
                elif color[nei] == color[node]:
                    return False
    return True

assert is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]])
assert not is_bipartite([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]])
print("Bipartite ")

Bipartite 


## More practice on LeetCode
Easy: Find the Town Judge (#997), Find Center of Star Graph (#1791), Flood Fill (#733)
Medium: Surrounded Regions (#130), Walls and Gates (#286), Number of Provinces (#547), Keys and Rooms (#841)

Also in `03-Interview Patterns/03.1`: Number of Islands, Rotting Oranges, Course Schedule.
Graph **algorithms** (topological sort, Dijkstra, MST) are in `02-Algorithms/02.4`.

## Key takeaways
- Use an **adjacency list** (`defaultdict(list)`).
- BFS = queue = shortest path (unweighted). DFS = stack/recursion = explore/connectivity.
- Always track **visited**. Grids are graphs with 4 neighbours.

**Next: `01.8-Tries.ipynb`**